# Predicting food inflation in Turkey

**Research question:** Do exchange rates and global food prices help predict monthly food inflation in Turkey?

This is our working notebook. It has not been executed. The initial cells download and inspect the three datasets, then align them by month. Model training and a final train/test split are deliberately left for later.

| Dataset | Role | Source |
| --- | --- | --- |
| Turkish food CPI and candidate food categories, 2025=100 | The Turkish price indices whose monthly changes we want to predict | [TÜİK](https://veriportali.tuik.gov.tr/tr/press/58289), distributed through [TCMB EVDS](https://evds3.tcmb.gov.tr/) |
| USD/TRY buying exchange rate | A possible predictor | [TCMB EVDS](https://evds3.tcmb.gov.tr/) |
| FAO Food Price Index and its food-group indices | Possible global-price predictors | [FAO monthly data](https://www.fao.org/worldfoodsituation/foodpricesindex/en/) |

We download candidate category series now so that we can inspect their coverage before choosing the final groups. Matching categories are related baskets, not identical products. For example, the Turkish dairy group includes eggs, and the oils/fats group is broader than FAO vegetable oils. Fuel and crude oil are outside the current scope.


## 1. Environment and access

Use Python 3.10 or newer. From the project directory, install the packages once in your chosen Python environment:

```bash
python -m pip install -r requirements.txt
python -m jupyterlab
```

Open this notebook and run its cells in order when ready. FAO needs no API key. For CPI and exchange rates, create a free [EVDS account](https://evds3.tcmb.gov.tr/) and copy the API key from your profile. The current [official API guide](https://evds3.tcmb.gov.tr/igmevdsms-dis/documents/showDocument?docId=16) explains this process.

Set `EVDS_API_KEY` in the environment that starts Jupyter, or enter it in the hidden prompt below. Do not paste the key into a cell or commit it. The key is sent only in the TCMB request header; it is not saved in download metadata.

The notebook has been checked statically, and its source URLs and current series codes were checked separately. Authenticated TCMB observation downloads have not been tested with a personal key.


In [ ]:
import csv
import hashlib
import io
import json
import os
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display


## 2. Download period and series

Start with January 2005, matching the reconstructed Turkish CPI history. Request data only through the end of the last completed calendar month. The APIs may have published fewer months; the coverage checks below will show that.

`2025=100` is the CPI reference scale, not its first observation year. We use the current CPI family throughout and do not splice it onto the archived 2003=100 family.


In [ ]:
project_candidates = [Path.cwd(), Path.cwd() / "project", *Path.cwd().parents]
project_dir = next(
    (
        candidate
        for candidate in project_candidates
        if (candidate / "handout" / "DSA101_Project_Proposal_Brief_2026Fall.pdf").is_file()
    ),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Open the notebook from the project directory or its parent.")

raw_dir = project_dir / "data" / "raw"
processed_dir = project_dir / "data" / "processed"
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

start_month = pd.Period("2005-01", freq="M")
end_month = pd.Timestamp.now().to_period("M") - 1
start_date = start_month.start_time
end_date = end_month.end_time.normalize()
downloaded_on = datetime.now(timezone.utc).strftime("%Y-%m-%d")

evds_base_url = "https://evds3.tcmb.gov.tr/igmevdsms-dis/"
fao_csv_url = (
    "https://www.fao.org/media/docs/worldfoodsituationlibraries/"
    "wfs-library/food_price_indices_data.csv?download=true&sfvrsn=523ebd2a_84"
)

cpi_series = {
    "TP.TUKFIY2025.01": "cpi_food",
    "TP.TUKFIY2025.0111": "cpi_cereals",
    "TP.TUKFIY2025.0112": "cpi_meat",
    "TP.TUKFIY2025.0114": "cpi_dairy_eggs",
    "TP.TUKFIY2025.0115": "cpi_oils_fats",
}
fx_series = {"TP.DK.USD.A.YTL": "usd_try"}
session = requests.Session()

display(pd.DataFrame({"start_month": [str(start_month)], "end_month": [str(end_month)]}))


## 3. Save original downloads

Each request is saved under `data/raw/`, with its URL, retrieval date and SHA-256 checksum in a separate metadata file. Existing snapshots from the same day and request are reused. Download metadata contains no API key.

The [EVDS guide](https://evds3.tcmb.gov.tr/igmevdsms-dis/documents/showDocument?docId=16) documents a maximum of 1,000 observations per request. The helper requests one calendar year at a time so a long daily exchange-rate series is not silently truncated. Request parameters belong in the EVDS URL path, and the key belongs in the HTTP header.


In [ ]:
def download_snapshot(url, name, suffix, headers=None):
    request_hash = hashlib.sha256(url.encode("utf-8")).hexdigest()[:12]
    destination = raw_dir / f"{name}_{downloaded_on}_{request_hash}.{suffix}"
    if destination.exists():
        return destination

    response = session.get(url, headers=headers, timeout=(15, 90))
    if headers and "key" in headers and response.status_code in {401, 403}:
        raise RuntimeError(
            "TCMB rejected the API key. Check your EVDS profile key or EVDS_API_KEY, then rerun this cell."
        )
    response.raise_for_status()
    if not response.content:
        raise ValueError(f"The {name} download was empty.")
    if "text/html" in response.headers.get("Content-Type", "").lower():
        raise ValueError(f"The {name} URL returned a web page instead of a data file.")

    destination.write_bytes(response.content)
    metadata = {
        "source_url": url,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "content_type": response.headers.get("Content-Type"),
        "bytes": len(response.content),
        "sha256": hashlib.sha256(response.content).hexdigest(),
    }
    destination.with_suffix(destination.suffix + ".metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    return destination


def evds_records(payload):
    records = payload.get("items") if isinstance(payload, dict) else payload
    if not isinstance(records, list) or not records:
        raise ValueError("EVDS did not return a non-empty observation or metadata list.")
    return records


def download_evds_years(series, name, api_key):
    records = []
    for year in range(start_date.year, end_date.year + 1):
        year_start = max(start_date, pd.Timestamp(year=year, month=1, day=1))
        year_end = min(end_date, pd.Timestamp(year=year, month=12, day=31))
        url = (
            f"{evds_base_url}series={'-'.join(series)}"
            f"&startDate={year_start:%d-%m-%Y}&endDate={year_end:%d-%m-%Y}&type=json"
        )
        path = download_snapshot(url, f"{name}_{year}", "json", headers={"key": api_key})
        payload = json.loads(path.read_text(encoding="utf-8-sig"))
        records.extend(evds_records(payload))
    return records


def parse_evds_observations(records, series, date_format):
    observations = pd.DataFrame(records)
    expected_columns = ["Tarih", *[name.replace(".", "_") for name in series]]
    missing_columns = set(expected_columns) - set(observations.columns)
    if missing_columns:
        raise ValueError(f"EVDS response is missing expected columns: {sorted(missing_columns)}")

    dates = pd.to_datetime(observations["Tarih"], format=date_format, errors="raise")
    if dates.duplicated().any():
        raise ValueError("The EVDS download contains duplicate dates; inspect the raw snapshots.")

    parsed = pd.DataFrame(index=pd.DatetimeIndex(dates, name="date"))
    for source_code, column_name in series.items():
        values = (
            observations[source_code.replace(".", "_")]
            .astype("string")
            .str.strip()
            .str.replace(",", ".", regex=False)
            .replace({"": pd.NA, "-": pd.NA})
        )
        parsed[column_name] = pd.to_numeric(values, errors="raise").to_numpy(dtype=float, na_value=float("nan"))
    parsed = parsed.sort_index()
    if (parsed <= 0).any().any():
        raise ValueError("A downloaded price index or exchange rate is zero or negative.")
    return parsed


## 4. Download and inspect FAO

FAO's CSV has title rows above its actual header and empty trailing columns. Find the header, retain the six named index series, and parse only monthly observations. Preserve the original download before doing this cleanup.


In [ ]:
fao_path = download_snapshot(fao_csv_url, "fao_food_price_indices", "csv")
csv_rows = list(csv.reader(io.StringIO(fao_path.read_text(encoding="utf-8-sig"))))
fao_header_row = next(
    (
        position
        for position, row in enumerate(csv_rows)
        if len(row) >= 7 and row[0].strip() == "Date" and row[1].strip() == "Food Price Index"
    ),
    None,
)
if fao_header_row is None:
    raise ValueError("FAO's CSV header changed. Inspect the saved raw file before proceeding.")

fao_raw = pd.read_csv(fao_path, skiprows=fao_header_row, usecols=range(7))
fao_raw.columns = fao_raw.columns.str.strip()
fao_columns = {
    "Food Price Index": "fao_food",
    "Meat": "fao_meat",
    "Dairy": "fao_dairy",
    "Cereals": "fao_cereals",
    "Oils": "fao_oils",
    "Sugar": "fao_sugar",
}
if not {"Date", *fao_columns}.issubset(fao_raw.columns):
    raise ValueError("FAO's expected index columns are missing.")

monthly_rows = fao_raw["Date"].astype("string").str.fullmatch(r"\d{4}-\d{2}").fillna(False)
fao_monthly_all = fao_raw.loc[monthly_rows, ["Date", *fao_columns]].copy()
fao_monthly_all["month"] = pd.to_datetime(fao_monthly_all.pop("Date"), format="%Y-%m")
fao_monthly_all = fao_monthly_all.set_index("month").rename(columns=fao_columns).sort_index()
fao_monthly_all = fao_monthly_all.apply(pd.to_numeric, errors="raise")
if fao_monthly_all.empty or fao_monthly_all.index.has_duplicates:
    raise ValueError("FAO has no monthly rows or contains duplicate months.")
if (fao_monthly_all <= 0).any().any():
    raise ValueError("A FAO price index is zero or negative.")

fao_monthly = fao_monthly_all.loc[start_date:end_date].copy()
display(fao_monthly_all.head())
display(fao_monthly_all.tail())


## 5. Confirm the Turkish series and provide the API key

The codes below were checked against the current EVDS catalogue. Download the public catalogue metadata again when running the notebook, confirm every selected code, and keep the metadata with the raw downloads.

The observation requests in the next sections require your API key. The hidden prompt does not print or save what you enter.


In [ ]:
selected_metadata = []
for group_code, requested_series in [
    ("bie_tukfiy2025", cpi_series),
    ("bie_dkdovytl", fx_series),
]:
    metadata_url = f"{evds_base_url}serieList/fe/type=json&code={group_code}"
    metadata_path = download_snapshot(metadata_url, f"metadata_{group_code}", "json")
    metadata_records = evds_records(json.loads(metadata_path.read_text(encoding="utf-8-sig")))
    metadata_frame = pd.DataFrame(metadata_records)
    if "SERIE_CODE" not in metadata_frame:
        raise ValueError("EVDS catalogue fields changed; inspect the saved metadata.")
    missing_codes = set(requested_series) - set(metadata_frame["SERIE_CODE"])
    if missing_codes:
        raise ValueError(f"Requested EVDS series are no longer in this catalogue: {sorted(missing_codes)}")
    selected_metadata.append(metadata_frame.loc[metadata_frame["SERIE_CODE"].isin(requested_series)])

series_metadata = pd.concat(selected_metadata, ignore_index=True)
display(series_metadata[["SERIE_CODE", "SERIE_NAME_ENG", "FREQUENCY_STR"]])

evds_api_key = os.environ.get("EVDS_API_KEY", "").strip()
if not evds_api_key:
    evds_api_key = getpass("EVDS API key, hidden: ").strip()
if not evds_api_key:
    raise ValueError("An EVDS API key is required for the Turkish CPI and USD/TRY downloads.")


## 6. Download Turkish CPI

These are monthly index levels, not prices in lira. Their percentage changes will give us Turkish inflation for each category. Inspect the actual first and last observations rather than assuming all selected categories have the same history.


In [ ]:
cpi_records = download_evds_years(cpi_series, "turkish_cpi", evds_api_key)
cpi_monthly = parse_evds_observations(cpi_records, cpi_series, "%Y-%m")
cpi_monthly.index = cpi_monthly.index.to_period("M").to_timestamp()
cpi_monthly.index.name = "month"

display(cpi_monthly.head())
display(cpi_monthly.tail())


## 7. Download USD/TRY and aggregate by month

Use the TCMB foreign-exchange buying rate, not the banknote buying rate. Request daily observations, then calculate the monthly mean of the published observations. Holidays and weekends can have no observation; we do not invent rates for those days.

Keep a count of the available daily rates in each month so that incomplete coverage remains visible.


In [ ]:
fx_records = download_evds_years(fx_series, "usd_try_daily", evds_api_key)
usd_try_daily = parse_evds_observations(fx_records, fx_series, "%d-%m-%Y")
usd_try_monthly = usd_try_daily.resample("MS").mean()
usd_try_monthly.index.name = "month"
usd_try_observation_counts = usd_try_daily["usd_try"].resample("MS").count().rename("published_daily_rates")

display(usd_try_daily.head())
display(usd_try_daily.tail())
display(usd_try_monthly.join(usd_try_observation_counts).tail())


## 8. Dataset sizes and missing observations

Report rows, columns, date ranges and missing values from the data we actually loaded. A daily FX row is one date; a CPI or FAO row is one month. These are different raw row counts and should not be added together as training examples.


In [ ]:
datasets = {
    "FAO, complete monthly download": fao_monthly_all,
    "FAO, requested period": fao_monthly,
    "Turkish CPI, requested period": cpi_monthly,
    "USD/TRY, daily": usd_try_daily,
    "USD/TRY, monthly mean": usd_try_monthly,
}
dataset_sizes = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": len(frame),
            "numeric_columns": len(frame.columns),
            "first_date": frame.index.min(),
            "last_date": frame.index.max(),
            "missing_values": int(frame.isna().sum().sum()),
        }
        for name, frame in datasets.items()
    ]
).set_index("dataset")

display(dataset_sizes)
display(cpi_monthly.isna().sum().rename("missing_CPI_values").to_frame())
display(fao_monthly.isna().sum().rename("missing_FAO_values").to_frame())


## 9. Align the three datasets by month

Create one row for each calendar month and keep missing observations visible. A missing month must not disappear before percentage changes are calculated; otherwise a multi-month change could be mislabeled as a one-month change.

This table aligns reference months only. It is **not yet a forecasting table**. A value labeled May may have been published in June. We must account for those publication delays when building prediction inputs later.


In [ ]:
calendar_months = pd.date_range(start_date, end_date, freq="MS", name="month")
monthly_levels = pd.concat(
    [cpi_monthly, usd_try_monthly, fao_monthly], axis=1
).reindex(calendar_months)
monthly_changes = monthly_levels.pct_change(fill_method=None).mul(100)

display(monthly_levels.head())
display(monthly_levels.tail())
display(monthly_changes.tail())


## 10. Coverage of candidate food-group comparisons

These pairings are candidate comparisons, not exact product matches. The counts below describe complete same-month percentage changes. They are **not final training row counts**: publication delays and lagged predictors will reduce or change the usable sample.

We will choose the final categories and evaluation dates after inspecting the results. An 80/20 split is a possible illustration, not a requirement. We should use earlier dates for training, evaluate one-month-ahead predictions on later dates, and keep model selection separate from the final test.


In [ ]:
category_pairs = {
    "overall food": ("cpi_food", "fao_food"),
    "cereals and cereal products": ("cpi_cereals", "fao_cereals"),
    "meat": ("cpi_meat", "fao_meat"),
    "milk, dairy and eggs": ("cpi_dairy_eggs", "fao_dairy"),
    "oils and fats": ("cpi_oils_fats", "fao_oils"),
}
coverage_rows = []
for category, (turkish_column, global_column) in category_pairs.items():
    comparison = monthly_changes[[turkish_column, "usd_try", global_column]].dropna()
    coverage_rows.append(
        {
            "category": category,
            "complete_change_months": len(comparison),
            "first_month": comparison.index.min(),
            "last_month": comparison.index.max(),
            "illustrative_80_percent": int(len(comparison) * 0.8),
            "illustrative_remaining_months": len(comparison) - int(len(comparison) * 0.8),
        }
    )
category_coverage = pd.DataFrame(coverage_rows).set_index("category")
display(category_coverage)


## 11. Save working tables

Raw snapshots remain unchanged. Save the aligned index levels, percentage changes and diagnostics separately. These CSVs can be loaded again when we work on the first-look figure and forecasting design.


In [ ]:
monthly_levels.to_csv(processed_dir / "monthly_index_levels.csv", index_label="month")
monthly_changes.to_csv(processed_dir / "monthly_percentage_changes.csv", index_label="month")
dataset_sizes.to_csv(processed_dir / "dataset_sizes.csv")
category_coverage.to_csv(processed_dir / "category_coverage.csv")
series_metadata.to_csv(processed_dir / "evds_series_metadata.csv", index=False)

print(f"Saved working tables in {processed_dir}")


## Next work

After running and inspecting the acquisition cells, we will:

1. Confirm source coverage, category definitions and reuse terms.
2. Choose a few category comparisons and produce one figure from our actual data for the proposal.
3. Document publication delays and build predictors that use only information available before each forecast month.
4. Choose chronological training, validation and final test periods from the resulting usable sample.
5. Compare an understandable baseline and regression models with inflation history alone, exchange rates, global prices, and both external predictors.

No model has been trained, no evaluation split has been selected, and no results are claimed in this notebook.

Sources and methods:

- [Current EVDS API guide](https://evds3.tcmb.gov.tr/igmevdsms-dis/documents/showDocument?docId=16), [CPI catalogue](https://evds3.tcmb.gov.tr/tumSeriler/2005/bie_tukfiy2025) and [FX catalogue](https://evds3.tcmb.gov.tr/tumSeriler/2501/bie_dkdovytl).
- [TÜİK CPI release, tables and methodology](https://veriportali.tuik.gov.tr/tr/press/58289).
- [FAO Food Price Index](https://www.fao.org/worldfoodsituation/foodpricesindex/en/).
- [EVDS reuse terms](https://evds3.tcmb.gov.tr/igmevdsms-dis/documents/showDocument?docId=18) and [FAO statistical database terms](https://www.fao.org/contact-us/terms/db-terms-of-use/en/).
- [Forecasting textbook: time-series cross-validation](https://otexts.com/fpp3/tscv.html).

Keep our course AI usage log as we develop the project.
